In [5]:
import os
import xml.etree.ElementTree as ET
import pandas as pd

def parse_semeval_folder(folder_path):
    records = []
    
    for filename in os.listdir(folder_path):
        if not filename.endswith('.xml'):
            continue
            
        filepath = os.path.join(folder_path, filename)
        tree = ET.parse(filepath)
        root = tree.getroot()
        
        question    = root.find('questionText').text.strip()
        reference   = root.find('referenceAnswers/referenceAnswer').text.strip()
        
        for student in root.findall('studentAnswers/studentAnswer'):
            answer = student.text.strip() if student.text else ""
            label  = student.get('accuracy')
            
            if not answer:
                continue
                
            records.append({
                'question'        : question,
                'reference_answer': reference,
                'student_answer'  : answer,
                'label'           : label
            })
    
    return pd.DataFrame(records)

# Paths — update these to match your folder structure
train_path = '../Data/semeval-3way/training/3way/sciEntsBank'
test_path  = '../Data/semeval-3way/test/3way/sciEntsBank/test-unseen-answers'

# Parse
df_train = parse_semeval_folder(train_path)
df_test  = parse_semeval_folder(test_path)

# Check
print(f"Train records : {len(df_train)}")
print(f"Train labels  :\n{df_train['label'].value_counts()}")
print(f"\nTest records  : {len(df_test)}")
print(f"Test labels   :\n{df_test['label'].value_counts()}")

Train records : 4969
Train labels  :
label
incorrect        2462
correct          2008
contradictory     499
Name: count, dtype: int64

Test records  : 540
Test labels   :
label
incorrect        249
correct          233
contradictory     58
Name: count, dtype: int64


In [6]:
df_train.to_csv('../data/semeval_train.csv', index=False)
df_test.to_csv('../data/semeval_test.csv', index=False)
print(f"Train: {len(df_train)} — {df_train['label'].value_counts().to_dict()}")
print(f"Test : {len(df_test)} — {df_test['label'].value_counts().to_dict()}")

Train: 4969 — {'incorrect': 2462, 'correct': 2008, 'contradictory': 499}
Test : 540 — {'incorrect': 249, 'correct': 233, 'contradictory': 58}


In [7]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
import os

In [8]:
with open('../outputs/error_profile.json', 'r') as f:
    error_profile = json.load(f)

print(f"Train : {len(df_train)} records")
print(f"Test  : {len(df_test)} records")
print(f"Error profile sections: {list(error_profile.keys())}")

Train : 4969 records
Test  : 540 records
Error profile sections: ['summary', 'substitutions', 'deletions', 'insertions', 'substitution_probs', 'position_analysis', 'word_length_analysis', 'punctuation_analysis']


In [9]:
import random

# Load substitution probabilities
sub_probs   = error_profile['substitution_probs']
del_chars   = error_profile['deletions']
ins_chars   = error_profile['insertions']

# Build deletion and insertion probability lists
del_char_list = list(del_chars.keys())
del_char_weights = list(del_chars.values())

ins_char_list = list(ins_chars.keys())
ins_char_weights = list(ins_chars.values())

# Error type probabilities from IAM
sub_rate = error_profile['summary']['substitution_rate']  # 0.6496
del_rate = error_profile['summary']['deletion_rate']      # 0.0713
ins_rate = error_profile['summary']['insertion_rate']     # 0.2791

def inject_noise(text, corruption_level, seed=42):
    """
    Inject OCR noise into clean text.
    corruption_level: float 0.0 to 1.0 (e.g. 0.1 = 10%)
    """
    random.seed(seed)
    chars  = list(text)
    result = []
    
    num_to_corrupt = int(len(chars) * corruption_level)
    indices_to_corrupt = set(random.sample(
        range(len(chars)), 
        min(num_to_corrupt, len(chars))
    ))
    
    for i, char in enumerate(chars):
        if i not in indices_to_corrupt:
            result.append(char)
            continue
        
        # Decide error type based on IAM rates
        error_type = random.choices(
            ['substitute', 'delete', 'insert'],
            weights=[sub_rate, del_rate, ins_rate]
        )[0]
        
        if error_type == 'substitute':
            char_lower = char.lower()
            if char_lower in sub_probs:
                targets = list(sub_probs[char_lower].keys())
                weights = list(sub_probs[char_lower].values())
                result.append(random.choices(targets, weights=weights)[0])
            else:
                result.append(char)
        
        elif error_type == 'delete':
            pass  # skip character — deletion
        
        elif error_type == 'insert':
            result.append(char)
            ins_char = random.choices(ins_char_list, weights=ins_char_weights)[0]
            result.append(ins_char)
    
    return ''.join(result)

# Test on a sample
sample_text = df_train['student_answer'].iloc[1]
print(f"Original : {sample_text}")
print(f"Noisy 10%: {inject_noise(sample_text, 0.1)}")
print(f"Noisy 20%: {inject_noise(sample_text, 0.2)}")
print(f"Noisy 30%: {inject_noise(sample_text, 0.3)}")

Original : Let the water evaporate and the salt is left behind.
Noisy 10%: Lat thenwater evaborate and the salt is teft be ind.
Noisy 20%: Lht thbsuater baaaorate and the sat is teft behiind.
Noisy 30%: Lyi tttater etraiorate andtthe saht as teft beh iid.


In [ ]:
print("Injecting noise into training set...")

df_train['noisy_10'] = df_train['student_answer'].apply(
    lambda x: inject_noise(x, 0.1))
df_train['noisy_20'] = df_train['student_answer'].apply(
    lambda x: inject_noise(x, 0.2))
df_train['noisy_30'] = df_train['student_answer'].apply(
    lambda x: inject_noise(x, 0.3))

print("Injecting noise into test set...")

df_test['noisy_10'] = df_test['student_answer'].apply(
    lambda x: inject_noise(x, 0.1))
df_test['noisy_20'] = df_test['student_answer'].apply(
    lambda x: inject_noise(x, 0.2))
df_test['noisy_30'] = df_test['student_answer'].apply(
    lambda x: inject_noise(x, 0.3))

# Save
df_train.to_csv('../data/semeval_train_noisy.csv', index=False)
df_test.to_csv('../data/semeval_test_noisy.csv', index=False)

print(f"\nDone!")
print(f"Train columns: {df_train.columns.tolist()}")
print(f"Test columns : {df_test.columns.tolist()}")